In [7]:
# chat-gpt

import re
from typing import List, Tuple

def generate_inflections(word: str) -> List[str]:
    """
    Generate possible inflected forms of a word, considering Spanish grammar.
    Handles singular and plural forms.
    """
    inflections = [word]
    if word.endswith(('a', 'e', 'i', 'o', 'u')):
        inflections.append(word + 's')
    elif word.endswith(('á', 'é', 'í', 'ó', 'ú')):
        inflections.append(word[:-1] + 'es')
    elif word.endswith('z'):
        inflections.append(word[:-1] + 'ces')
    else:
        inflections.append(word + 'es')
    return inflections

def build_inflected_patterns(expression: str) -> List[str]:
    """
    Build regex patterns for the given expression, including inflected forms for each word.
    """
    words = expression.split()
    if len(words) == 1:
        # Single word case
        return [re.escape(inflected) for inflected in generate_inflections(expression)]
    else:
        # Multiword expression case
        inflected_forms = [generate_inflections(word) for word in words]
        # Build regex patterns for all combinations of inflected forms
        patterns = []
        for combo in zip(*inflected_forms):
            patterns.append(r' '.join(map(re.escape, combo)))
        return patterns

def find_offsets_and_forms(text: str, expression: str) -> List[Tuple[Tuple[int, int], str]]:
    """
    Find occurrences of the word or multiword expression in the text.
    Ignores punctuation and considers plural inflections.
    Returns the offsets and the exact matched form.
    """
    # Generate patterns considering inflections
    patterns = build_inflected_patterns(expression)
    # Allow patterns to ignore punctuation around words
    punctuation_aware_pattern = r'\b(?:{})\b'.format('|'.join(patterns))
    
    matches = []
    for match in re.finditer(punctuation_aware_pattern, text, re.IGNORECASE):
        matched_text = match.group()  # Includes the exact matched form
        # Extract offsets for the exact form
        start, end = match.span()
        matches.append(((start, end), matched_text))
    return matches

# ejemplo 1
texto1 = "¡Los gatos son animales interesantes! Los perros también son interesantes, e INTERESANTES. Mi tortuga no es tan interesante."
palabra = "interesante"
resultados_palabra = find_offsets_and_forms(texto1, palabra)

print('EJEMPLO 1')
print(f'Texto: {texto1}')
print(f'Búsqueda: {palabra}')
print(f'Output: {resultados_palabra}')

print('\n')

texto2 = 'Vamos a hablar de los derechos laborales. El derecho laboral es...'
expresion = 'derecho laboral'
resultados_expresion = find_offsets_and_forms(texto2, expresion)

print('EJEMPLO 2')
print(f'Texto: {texto2}')
print(f'Búsqueda: {expresion}')
print(f'Output: {resultados_expresion}')


EJEMPLO 1
Texto: ¡Los gatos son animales interesantes! Los perros también son interesantes, e INTERESANTES. Mi tortuga no es tan interesante.
Búsqueda: interesante
Output: [((24, 36), 'interesantes'), ((61, 73), 'interesantes'), ((77, 89), 'INTERESANTES'), ((112, 123), 'interesante')]


EJEMPLO 2
Texto: Vamos a hablar de los derechos laborales. El derecho laboral es...
Búsqueda: derecho laboral
Output: [((22, 40), 'derechos laborales'), ((45, 60), 'derecho laboral')]


In [6]:
# chat-gpt

def filter_offsets_across_words(data):
    """
    Filters offsets across words, ensuring no offset for a word is included in the offsets of another word.

    Args:
        data (dict): Dictionary with words as keys and a list of (start, end) tuples as values.

    Returns:
        dict: A dictionary with filtered offsets where no offset is included in the offsets of another word.
    """
    filtered_data = {}
    
    for current_word, current_offsets in data.items():
        filtered_offsets = []
        
        for current_offset in current_offsets:
            start1, end1 = current_offset
            is_contained = False
            
            for other_word, other_offsets in data.items():
                if current_word == other_word:
                    continue  # Skip checking against the same word
                
                for start2, end2 in other_offsets:
                    # Check if the current offset is fully contained within another offset
                    if start1 >= start2 and end1 <= end2:
                        is_contained = True
                        # Only keep the longer one (current vs other)
                        if end2 - start2 > end1 - start1:
                            break
            
            # Add the offset only if it is not contained within any other
            if not is_contained:
                filtered_offsets.append(current_offset)
        
        # Store the filtered offsets for the current word
        filtered_data[current_word] = filtered_offsets

    return filtered_data

ejemplo = {
    "riesgos laborales": [(246, 263)],
    "riesgos": [(246, 253), (146, 153)],
}

results = filter_offsets_across_words(ejemplo)

print(f"input: {ejemplo}")
print(f"output: {results}")

input: {'riesgos laborales': [(246, 263)], 'riesgos': [(246, 253), (146, 153)]}
output: {'riesgos laborales': [(246, 263)], 'riesgos': [(146, 153)]}
